In [2]:
import arcpy
from arcpy.sa import Reclassify, RemapRange, Raster, SetNull, ExtractByMask
import os
from arcgis.gis import GIS 

# Allow ArcPy tools to overwrite existing outputs
arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

# Define a local directory where input data is stored
path_folder = r"C:\path\to\repo\root"

data_dir = os.path.join(path_folder, "data")
print(f"Data directory set to: {data_dir}")

# Define a local directory where all outputs will be stored
out_dir = os.path.join(path_folder, "outputs")

# Define the file geodatabase that will store all derived outputs
out_gdb = os.path.join(out_dir, "outputs.gdb")

# Create the outputs directory (safe to call repeatedly)
os.makedirs(out_dir, exist_ok=True)

# Create a fresh file geodatabase
arcpy.management.CreateFileGDB(out_dir, "outputs.gdb")

# Set the workspace so all subsequent outputs are written to this geodatabase
arcpy.env.workspace = out_gdb

# Connect to ArcGIS Online (or your portal) using your credentials
gis = GIS('home')

Data directory set to: C:\Users\alicia.canales\Documents\ArcGIS\Projects\GDE\data1


##### Fetch land polygon feature layer item to be used for area calculation later

In [17]:
feature_layer_url = "https://services2.arcgis.com/C8EMgrsFcRFL6LrL/arcgis/rest/services/Land_Polygon_(from_Natural_Earth)/FeatureServer/0"

arcpy.conversion.FeatureClassToFeatureClass(
    in_features=feature_layer_url,
    out_path=out_gdb,
    out_name="land_polygon"
)
print("Done: saved as outputs.gdb/land_polygon") 

Done: saved as outputs.gdb/land_polygon


##### Project Global Aridity Index Data

In [3]:
# Input raster path (Global Aridity Index v3, annual average, no Antarctica)
in_raster = os.path.join(
    data_dir,
    "Global-AI_ET0_annual_v3",
    "Global-AI_ET0_v3_annual",
    "ai_v3_yr.tif"
)

# Output raster name inside outputs.gdb
out_raster_name = "ai_v3_yr_cyl_eqarea.tif" # Global Aridity Index raster projected to equal-area CRS
out_raster = os.path.join(out_dir, out_raster_name)

# Describe input raster to use its defined coordinate system and resolution
in_desc = arcpy.Describe(in_raster)

# Target equal-area projection for area calculations (World Cylindrical Equal Area)
# Using the Esri factory code for World Cylindrical Equal Area (54034)
out_spatial_ref = arcpy.SpatialReference(54034)

# Project raster to equal-area CRS for downstream area calculations
arcpy.management.ProjectRaster(
    in_raster=in_raster,
    out_raster=out_raster,
    out_coor_system=out_spatial_ref,
    resampling_type="NEAREST",
    geographic_transform=None,
    Registration_Point=None,
    in_coor_system=in_desc.spatialReference,
    vertical="NO_VERTICAL",
)

print("Projected raster saved as:", out_raster)

Projected raster saved as: C:\Users\alicia.canales\Documents\ArcGIS\Projects\GDE\data1\outputs\ai_v3_yr_cyl_eqarea.tif


##### Remove any remaining Aridity cells that overlap Antarctica

In [13]:
# Define inputs
in_features = "land_polygon"  # feature class/layer that includes Antarctica polygons

oid_list = [51,52,53,54,55,56,57,58,59,60,121,122,123,124,125,207,209,211,212,213,214,
            215,216,217,218,219,220,221,222,224,225,226,227,228,229,230,231,232,233,234,
            235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,
            254,255,256,257,258,259,260,261,262,264,265,266,267,268,269,270,271,272,273,
            274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,
            293,294,295,296,297,1379]

in_raster = Raster(os.path.join(out_dir, "ai_v3_yr_cyl_eqarea.tif"))

# Buffer distance 
buffer_distance = "100 Miles"

# Define outputs 
buffer_fc = os.path.join(out_gdb, "antarctica_buffer")          # feature class
out_raster = os.path.join(out_dir, "ai_v3_yr_no_antarctica.tif")  # tif output in folder

# Build definition query for Antarctica OIDs
oid_field = arcpy.Describe(in_features).OIDFieldName
oid_field_sql = arcpy.AddFieldDelimiters(in_features, oid_field)
where_clause = f"{oid_field_sql} IN ({','.join(map(str, oid_list))})"

antarctica_lyr = "antarctica_lyr"
arcpy.management.MakeFeatureLayer(in_features, antarctica_lyr, where_clause)

sel_count = int(arcpy.management.GetCount(antarctica_lyr)[0])

# Buffer Antarctica polygons
if arcpy.Exists(buffer_fc):
    arcpy.management.Delete(buffer_fc)

print(f"Creating buffer: {buffer_distance}")
arcpy.analysis.Buffer(
    in_features=antarctica_lyr,
    out_feature_class=buffer_fc,
    buffer_distance_or_field=buffer_distance,
    dissolve_option="ALL",
    method="GEODESIC" 
)

buf_count = int(arcpy.management.GetCount(buffer_fc)[0])
print(f"Buffered feature count: {buf_count}")

# Extract raster OUTSIDE the buffered Antarctica polygon
if arcpy.Exists(out_raster):
    arcpy.management.Delete(out_raster)

print("Extracting raster OUTSIDE Antarctica buffer (Antarctica + buffer => NoData)...")
with arcpy.EnvManager(
    snapRaster=in_raster,
    cellSize=in_raster,
    extent=in_raster.extent
):
    out = ExtractByMask(
        in_raster=in_raster,
        in_mask_data=buffer_fc,
        extraction_area="OUTSIDE"
    )

out.save(out_raster)
print("Done. Output raster:", out_raster)

# Optional cleanup
arcpy.management.Delete(buffer_fc)

Creating buffer: 100 Miles
Buffered feature count: 1
Extracting raster OUTSIDE Antarctica buffer (Antarctica + buffer => NoData)...
Done. Output raster: C:\Users\alicia.canales\Documents\ArcGIS\Projects\GDE\data1\outputs\ai_v3_yr_no_antarctica.tif


<Result 'true'>

##### Reclassify Global Aridity Index Data

In [14]:
# Name of the projected equal-area raster
projected_raster = os.path.join(out_dir,"ai_v3_yr_no_antarctica.tif")

# Name for the output binary desert-mask raster
reclass_raster = "ai_v3_yr_cyl_eqarea_reclass"

# Conversion factor applied to the raster's Value field to create AI_conv
scale_factor = 10000

# Define remap values for reclassify
remap_range = RemapRange(
    [[0.0001, 0.200, 1]]
)

print("Starting: build raster attribute table...")
arcpy.management.BuildRasterAttributeTable(projected_raster, "OVERWRITE")
print("Done: raster attribute table built.")

print("Starting: add field AI_conv...")
arcpy.management.AddField(projected_raster, "AI_conv", "DOUBLE")
print("Done: field AI_conv added.")


arcpy.management.CalculateField(
    in_table=projected_raster,
    field="AI_conv",
    expression=f"!Value! / ({scale_factor})",
    expression_type="PYTHON3",
)
print("Done: AI_conv calculated.")

print("Starting: reclassify AI_conv to binary mask")
out_mask = Reclassify(
    in_raster=projected_raster,
    reclass_field="AI_conv",
    remap=remap_range, # 
    missing_values="NODATA",
)

# Save as TIFF
tiff_path = os.path.join(out_dir, f"{reclass_raster}.tif")
if arcpy.Exists(tiff_path):
    arcpy.management.Delete(tiff_path)

print(f"Saving as TIFF: {tiff_path}")
out_mask.save(tiff_path)

Starting: build raster attribute table...
Done: raster attribute table built.
Starting: add field AI_conv...
Done: field AI_conv added.
Done: AI_conv calculated.
Starting: reclassify AI_conv to binary mask
Saving as TIFF: C:\Users\alicia.canales\Documents\ArcGIS\Projects\GDE\data1\outputs\ai_v3_yr_cyl_eqarea_reclass.tif


##### Project GLWD 

In [35]:
in_raster = os.path.join(
    data_dir,
    "GLWD_v2_0_combined_classes_tif",
    "GLWD_v2_0_combined_classes",
    "GLWD_v2_0_main_class.tif",
)

# Output name inside outputs.gdb
out_raster_name = "glwd_inland_cyl_eqarea"
out_raster = os.path.join(arcpy.env.workspace, out_raster_name)

# Equal-area projection for area-preserving calculations
out_spatial_ref = arcpy.SpatialReference(54034)  # World Cylindrical Equal Area (Esri:54034)

resampling_type = "NEAREST"

# Use the input raster's defined coordinate system
in_desc = arcpy.Describe(in_raster)

print("Starting: ProjectRaster (GLWD -> World Cylindrical Equal Area)")

arcpy.management.ProjectRaster(
    in_raster=in_raster,
    out_raster=out_raster,
    out_coor_system=out_spatial_ref,
    resampling_type=resampling_type,
    geographic_transform=None,
    Registration_Point=None,
    in_coor_system=in_desc.spatialReference,
    vertical="NO_VERTICAL",
)

print("Done: Project Raster complete")

Starting: ProjectRaster (GLWD -> World Cylindrical Equal Area)
Done: Project Raster complete


##### Reclassify GLWD

In [36]:
# Reclassify GLWD raster to binary mask of inland wetlands (GLWD classes 20–27 and 32)
in_raster ="glwd_inland_cyl_eqarea"
out_name = os.path.join(out_dir, "glwd_inland_wetlands_reclass.tif")

glwd_remap = RemapRange([
    [20, 27, 1],  # GLWD classes 20–27 -> wetlands (1)
    [32, 32, 1],  # GLWD class 32 -> wetlands (1)
])

print("Starting: Reclassify...)")
out_raster = Reclassify(
    in_raster= in_raster,
    reclass_field="Value",
    remap=glwd_remap,
    missing_values="NODATA"   # all other classes -> NoData
)
out_raster.save(out_name)

print("Done: GLWD inland wetlands reclass output saved.")

Done: GLWD inland wetlands reclass output saved.


##### Calculate desert area

In [3]:
# Defie input raster for area calculation
in_table = os.path.join(out_dir, "ai_v3_yr_cyl_eqarea_reclass.tif")

arcpy.management.BuildRasterAttributeTable(in_table, "OVERWRITE")
print("Done: raster attribute table built.")

# Get raster cell size from the dataset to avoid hard-coding resolution
r_desc = arcpy.Describe(in_table)
cell_area_m2 = r_desc.meanCellWidth * r_desc.meanCellHeight

# Add an output field to store area in square kilometers (km²)
arcpy.management.AddField(in_table, "Area_km2", "DOUBLE")
print("Done: Area_km2 field added.")

# Calculate Area_km2 using raster attribute table Count:
# Area_km2 = Count * cell_area_m2 / 1,000,000  (convert m² to km²)
expression = f"(!Count! * {cell_area_m2}) / 1000000"

print("Starting: calculate Area_km2...")
arcpy.management.CalculateField(
    in_table=in_table,
    field="Area_km2",
    expression=expression,
    expression_type="PYTHON3"
)
print("Done: Area_km2 calculated.")

Done: raster attribute table built.
Done: Area_km2 field added.
Starting: calculate Area_km2...
Done: Area_km2 calculated.


##### Create summary table for desert area

In [6]:
# Define input and output table names
in_table = "ai_v3_yr_cyl_eqarea_reclass.tif"
out_table = os.path.join(arcpy.env.workspace, "ai_desert_total_area_km2")

# Define statistics fields as a list of lists, where each inner list contains the field name and the statistic type
statistics_fields = [["Area_km2", "SUM"]]

# Summarize total of global deserts area (km²)
arcpy.analysis.Statistics(
    in_table=in_table,      # Input table
    out_table=out_table,  # Output table
    statistics_fields=statistics_fields,    # Sum Area_km2
    case_field=None,                                     
    concatenation_separator=""                           
)
print("Done: Statistics Table complete.")

Done: Statistics Table complete.


##### Overlay inland wetlands on top of desert mask

In [9]:
# Clip the GLWD raster to arid/semi-arid mask (keep only cells inside the mask)
wetland_raster = os.path.join(out_dir,"glwd_inland_wetlands_reclass.tif")
desert_mask = os.path.join(out_dir,"ai_v3_yr_cyl_eqarea_reclass.tif")

# Define output path for the ExtractByMask raster
out_name = os.path.join(out_dir,"wetlands_within_deserts.tif")

# Use EnvManager so snapRaster and cellSize apply ONLY within this block
with arcpy.EnvManager(
    workspace= arcpy.env.workspace,
    snapRaster= wetland_raster,   # align output grid to wetlands raster
    cellSize= wetland_raster      # use wetlands raster resolution
):
    out_raster = arcpy.sa.ExtractByMask(
        in_raster= wetland_raster,   
        in_mask_data= desert_mask 
    )
    out_raster.save(out_name)
    del out_raster

##### Project global land area polygon

In [18]:
# Inputs
in_dataset = os.path.join(out_gdb, "land_polygon")

# Intermediate (land polygons with Antarctica removed)
no_antarctica_fc = os.path.join(out_gdb, "land_polygon_no_antarctica")

# Final projected output
out_dataset = os.path.join(out_gdb, "land_polygon_cyl_eqarea")

# Using the Esri factory code for World Cylindrical Equal Area (54034)
out_spatial_ref = arcpy.SpatialReference(54034)

# ObjectIDs to remove
antarctica_oid_list = [
    51,52,53,54,55,56,57,58,59,60,121,122,123,124,125,207,209,211,212,213,214,
    215,216,217,218,219,220,221,222,224,225,226,227,228,229,230,231,232,233,234,
    235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,
    254,255,256,257,258,259,260,261,262,264,265,266,267,268,269,270,271,272,273,
    274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,
    293,294,295,296,297,1379
]

# Clean up old outputs
for fc in (no_antarctica_fc, out_dataset):
    if arcpy.Exists(fc):
        arcpy.management.Delete(fc)

# Build a "definition query" (where clause) for the OIDs
oid_field = arcpy.Describe(in_dataset).OIDFieldName
oid_field_sql = arcpy.AddFieldDelimiters(in_dataset, oid_field)
where_clause = f"{oid_field_sql} IN ({','.join(map(str, antarctica_oid_list))})"

# Make a layer, select Antarctica OIDs, then switch selection to KEEP everything else
land_lyr = "land_lyr"
arcpy.management.MakeFeatureLayer(in_dataset, land_lyr)
arcpy.management.SelectLayerByAttribute(land_lyr, "NEW_SELECTION", where_clause)

# Switch selection => selects all NON-Antarctica polygons
arcpy.management.SelectLayerByAttribute(land_lyr, "SWITCH_SELECTION")
kept_count = int(arcpy.management.GetCount(land_lyr)[0])
print(f"Polygons kept (non-Antarctica): {kept_count}")

# Write the non-Antarctica polygons to a new feature class
arcpy.management.CopyFeatures(land_lyr, no_antarctica_fc)
print("Done: Antarctica removed:", no_antarctica_fc)

# Project the cleaned feature class to 54034
arcpy.management.Project(
    in_dataset=no_antarctica_fc,
    out_dataset=out_dataset,
    out_coor_system=out_spatial_ref,
    preserve_shape="NO_PRESERVE_SHAPE",
    max_deviation=None,
    vertical="NO_VERTICAL"
)
print("Done: Project land polygon (no Antarctica) complete:", out_dataset)

# Clean up intermediate data
arcpy.management.Delete(no_antarctica_fc)

Polygons kept (non-Antarctica): 1317
Done: Antarctica removed: C:\Users\alicia.canales\Documents\ArcGIS\Projects\GDE\data1\outputs\outputs.gdb\land_polygon_no_antarctica
Done: Project land polygon (no Antarctica) complete: C:\Users\alicia.canales\Documents\ArcGIS\Projects\GDE\data1\outputs\outputs.gdb\land_polygon_cyl_eqarea


<Result 'true'>

##### Calculate global land area

In [19]:
in_table = "land_polygon_cyl_eqarea"   # Global land polygon (exclude Antarctica) (Equal-Area projection)
out_table = os.path.join(arcpy.env.workspace, "global_land_area_km2")  # Output table for total land area in km²
statistics_fields = [["Area_km2", "SUM"]]  # Define statistics fields as a list of lists    

# Using the Esri factory code for World Cylindrical Equal Area (54034)
out_spatial_ref = arcpy.SpatialReference(54034)

# Add an output field to store area in square kilometers (km²)
arcpy.management.AddField(in_table, "Area_km2", "DOUBLE")
print("Done: Area_km2 field added.")

# Calculate geometry field
geometry_property = [["Area_km2", "AREA"]]
arcpy.management.CalculateGeometryAttributes(
    in_features= in_table,
    geometry_property= geometry_property,
    length_unit="",
    area_unit="SQUARE_KILOMETERS",
    coordinate_system= out_spatial_ref,
    coordinate_format="SAME_AS_INPUT"
)
print("Done: Area_km2 calculated.")

# Calculate summary table
arcpy.analysis.Statistics(
    in_table=in_table,   # Global land polygon (exclude Antarctica) (Equal-Area projection)
    out_table=out_table,
    statistics_fields=statistics_fields,
    case_field=None,
    concatenation_separator=""
)
print("Done: Statistics calculated.")

Done: Area_km2 field added.
Done: Area_km2 calculated.
Done: Statistics calculated.


##### Calculate area of inland wetlands

In [12]:
# Define input raster for area calculation
in_table = os.path.join(out_dir,"wetlands_within_deserts.tif") # inland wetlands raster
expression = f"!Count! * ({cell_area_m2}) / 1000000"  # Convert count to area in km²

# Get raster cell size from the dataset to avoid hard-coding resolution
r_desc = arcpy.Describe(in_table)
cell_area_m2 = r_desc.meanCellWidth * r_desc.meanCellHeight

# Add an output field to store area in square kilometers (km²)
arcpy.management.AddField(in_table, "Area_km2", "DOUBLE")
print("Done: Area_km2 field added.")

# Calculate Area_km2 using raster attribute table Count:
# Area_km2 = Count * cell_area_m2 / 1,000,000  (convert m² to km²)
expression = f"(!Count! * {cell_area_m2}) / 1000000"

arcpy.management.CalculateField(
    in_table=in_table, # inland wetlands raster
    field="Area_km2",  
    expression=expression,
    expression_type="PYTHON3",
    code_block="",
    field_type="DOUBLE",
    enforce_domains="NO_ENFORCE_DOMAINS"
)
print("Done: Area_km2 calculated.")

Done: Area_km2 field added.
Done: Area_km2 calculated.


##### Report out percantage calculations 

In [22]:
#Total desert Area (km2)
tbl = os.path.join(out_dir,"ai_v3_yr_cyl_eqarea_reclass.tif")
field_name = "Area_km2"
with arcpy.da.SearchCursor(tbl, [field_name]) as cursor:
    for row in cursor:
        d_value = row[0]
    print(f"Area: {d_value} km²")
    
desert_area = d_value

# Global land area (km2)
tbl = os.path.join(out_gdb,"global_land_area_km2")
field_name = "SUM_Area_km2"
with arcpy.da.SearchCursor(tbl, [field_name]) as cursor:
    for row in cursor:
        gl_value = row[0]
    print(f"Area: {gl_value} km²")
global_land_area = gl_value

# Wetlands within desert area (km2)
tbl = os.path.join(out_dir,"wetlands_within_deserts.tif")
field_name = "Area_km2"
with arcpy.da.SearchCursor(tbl, [field_name]) as cursor:
    for row in cursor:
        w_value = row[0]
    print(f"Area: {w_value} km²")
    
wetlands_within_deserts = w_value

Area: 36035495.081 km²
Area: 134381791.2314885 km²
Area: 762534.671344 km²


In [23]:
# 1. % desert area covered by inland wetlands
pct = (wetlands_within_deserts/desert_area)*100
print(f"% desert area covered by inland wetlands: {round(pct,2)}")

# 2. % global land area covered by inland desert wetlands
pct_1 = (wetlands_within_deserts/global_land_area) * 100
print(f"% global land area covered by inland wetlands: {round(pct_1,2)}")

print(f"Total area (km²) of inland wetlands within arid + hyper‑arid zones: {round(wetlands_within_deserts,2):,}")
print(f"Total area (km²) of the desert region: {round(desert_area,2):,}")

% desert area covered by inland wetlands: 2.12
% global land area covered by inland wetlands: 0.57
Total area (km²) of inland wetlands within arid + hyper‑arid zones: 762,534.67
Total area (km²) of the desert region: 36,035,495.08
